In [0]:
import polars as pl
from mlforecast import MLForecast
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import bias, mae, rmse

from config import config
from src.utils import CompetitionMetric

In [0]:
# Load preprocessed dataset and convert to pandas
df = (
    pl.read_parquet("data/preprocessed/sales.parquet")
    .to_pandas()
    .assign(
        Client=lambda df: df["Client"].astype("int"),
        Product=lambda df: df["Product"].astype("int"),
        Warehouse=lambda df: df["Warehouse"].astype("int"),
    )
)

In [0]:
from config import config_project
from pyspark.sql.functions import current_timestamp, to_utc_timestamp
import pandas as pd
from pyspark.sql import SparkSession

catalog = config_project["catalog"]
schema = config_project["schema"]

def save_to_catalog(df_processed: pd.DataFrame, spark: SparkSession, config: dict):
    """
    Save the processed DataFrame into a Databricks table with a timestamp and enable Change Data Feed.

    Parameters:
    df_processed (pd.DataFrame): The processed DataFrame to be saved.
    spark (SparkSession): The Spark session to use for saving the DataFrame.
    """

    df_processed_with_timestamp = spark.createDataFrame(df_processed).withColumn(
        "update_timestamp_utc", to_utc_timestamp(current_timestamp(), "UTC")
    )

    df_processed_with_timestamp.write.mode("overwrite").saveAsTable(
        f"{config['catalog']}.{config['schema']}.processed_data"
    )

    spark.sql(
        f"ALTER TABLE {config['catalog']}.{config['schema']}.processed_data "
        "SET TBLPROPERTIES (delta.enableChangeDataFeed = true);"
    )

spark = SparkSession.builder.getOrCreate()

save_to_catalog(df_processed=df, spark=spark, config=config_project)

In [0]:
# Extract configuration details
static_features = config["fit"]["static_features"]
reference_features = config["reference_features"]

# Initialize the forecast model
mlf = MLForecast(**config["init"])

# Perform cross validation
cv_df = mlf.cross_validation(df[static_features + reference_features], **config["fit"])

# Aggregate metrics from cross validation results
cv_evaluate = evaluate(
    cv_df.drop(columns="cutoff"),
    metrics=[rmse, mae, CompetitionMetric, bias],
    agg_fn="mean",
)

# Create a dictionary with aggregated evaluation metrics
dict_metrics = {k: float(i) for k, i in cv_evaluate.set_index("metric")["lgb"].items()}
dict_metrics